<a href="https://colab.research.google.com/github/kanakadurgachilukuri4-ui/Multi_Query_RAG_System/blob/main/Multi_Query_RAG_System_162.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.0 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

**Create sample knowledge-base**

In [ ]:
documents = [
    """
    Transformer models are deep learning architectures designed primarily for
    processing sequential data. They were introduced in the paper "Attention Is
    All You Need". Unlike traditional recurrent neural networks, Transformers
    process many tokens in parallel, making training more efficient.
    """,

    """
    A major advantage of Transformer models is the self-attention mechanism.
    Self-attention allows the model to determine how important each word is
    relative to other words in the input. This helps Transformers capture
    relationships between words even when they are far apart in a sentence.
    """,

    """
    Transformers are highly parallelizable because they do not need to process
    tokens strictly one after another. This makes them faster to train on modern
    hardware such as GPUs and TPUs compared with many recurrent neural networks.
    """,

    """
    Transformer models can handle long-range dependencies effectively.
    Self-attention enables a Transformer to directly connect different parts
    of a sequence, which is useful for understanding the context of long
    sentences and documents.
    """,

    """
    Transformers have been successfully applied to many areas of artificial
    intelligence. Applications include machine translation, text generation,
    question answering, text summarization, sentiment analysis, speech
    processing, image recognition, and multimodal AI systems.
    """,

    """
    Compared with RNNs, Transformers generally provide better scalability and
    parallel processing capabilities. RNNs process sequences step by step,
    while Transformers can process multiple tokens simultaneously. This makes
    Transformers particularly suitable for large-scale language models.
    """,

    """
    One limitation of standard Transformer models is their computational cost.
    The self-attention mechanism can require substantial memory and computation
    when processing very long sequences. Researchers have therefore developed
    efficient attention mechanisms and other techniques to handle longer inputs.
    """,

    """
    Transformer architectures form the foundation of many modern language
    models. Models such as BERT, GPT, T5, and other large language models use
    Transformer-based architectures. These models have significantly improved
    natural language processing capabilities.
    """
]

print("Number of documents:", len(documents))

Number of documents: 8


**Load the embedding model**

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  #all-MiniLM-L6-v2 produces 384-dimensional embeddings
print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


**Create Embeddings for the Documents**

In [ ]:
document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)
print("Embedding shape:", document_embeddings.shape)

Embedding shape: (8, 384)


**Create the FAISS Vector Index**

In [ ]:
embedding_dimension = document_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dimension)
faiss_index.add(document_embeddings)
print("FAISS index created successfully!")
print("Number of vectors in index:", faiss_index.ntotal)

FAISS index created successfully!
Number of vectors in index: 8


**Create a Basic Retrieval Function**

Before implementing multiple queries, let's create a function that can retrieve the most relevant documents for one query.

In [ ]:
def retrieve_documents(query, top_k=3):
    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )
    # Search the FAISS index
    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )
    # Retrieve the corresponding documents
    retrieved_documents = [
        documents[index]
        for index in indices[0]
    ]
    return retrieved_documents, distances[0]

**Test Single-Query Retrieval**

Before moving to Multi-Query RAG, let's verify that our retrieval system actually works.

In [ ]:
user_query = "What are the advantages of transformer models?"
retrieved_documents, distances = retrieve_documents(
    user_query,
    top_k=3
)
print("User Query:")
print(user_query)
print("\nRetrieved Documents:")
for i, (doc, distance) in enumerate(
    zip(retrieved_documents, distances), start=1
):
    print(f"\n--- Document {i} ---")
    print(doc.strip())
    print(f"Distance: {distance:.4f}")

User Query:
What are the advantages of transformer models?

Retrieved Documents:

--- Document 1 ---
A major advantage of Transformer models is the self-attention mechanism.
    Self-attention allows the model to determine how important each word is
    relative to other words in the input. This helps Transformers capture
    relationships between words even when they are far apart in a sentence.
Distance: 0.4997

--- Document 2 ---
Transformer models are deep learning architectures designed primarily for
    processing sequential data. They were introduced in the paper "Attention Is
    All You Need". Unlike traditional recurrent neural networks, Transformers
    process many tokens in parallel, making training more efficient.
Distance: 0.7351

--- Document 3 ---
Transformers are highly parallelizable because they do not need to process
    tokens strictly one after another. This makes them faster to train on modern
    hardware such as GPUs and TPUs compared with many recurrent neura

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from getpass import getpass
api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)
print("Gemini client configured successfully!")

Enter your Gemini API key: ··········
Gemini client configured successfully!


**Gemini Multi-Query Generator**

We'll create the function that asks Gemini to generate 4 diverse search queries from the original question.

In [ ]:
import json
def generate_queries(user_query, num_queries=4):
    prompt = f"""
You are a search-query generation component of a Multi-Query RAG system.
The user asked:
"{user_query}"
Generate exactly {num_queries} different search queries that can retrieve
useful information for answering the user's question.
Requirements:
- Each query should explore a different aspect of the question.
- Queries should be concise and suitable for semantic search.
- Do not answer the question.
- Do not repeat the original question.
- Do not add numbering.
- Return only the requested JSON object.
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": {
                "type": "object",
                "properties": {
                    "queries": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": ["queries"]
            }
        }
    )
    result = json.loads(response.text)
    return result["queries"][:num_queries]

**Generate and Display Multiple Queries**

**Retrieve Documents for Each Generated Query**

In [ ]:
def multi_query_retrieval(generated_queries, top_k=3):
    all_retrieved_documents = []
    for query in generated_queries:
        # Convert query into an embedding
        query_embedding = embedding_model.encode(
            [query],
            convert_to_numpy=True
        )
        # Search FAISS
        distances, indices = faiss_index.search(
            query_embedding,
            top_k
        )
        # Store retrieved documents
        for index, distance in zip(indices[0], distances[0]):
            all_retrieved_documents.append({
                "query": query,
                "document": documents[index],
                "distance": distance
            })
    return all_retrieved_documents

**Run Multi-Query Retrieval**

In [ ]:
generated_queries = [
    "What is Retrieval Augmented Generation?",
    "How does Retrieval Augmented Generation work?",
    "What are the applications of Retrieval Augmented Generation?"
]
def multi_query_retrieval(generated_queries, top_k=3):

    all_results = []

    for query in generated_queries:

        # Convert query into an embedding
        query_embedding = embedding_model.encode(
            [query],
            convert_to_numpy=True
        )

        # Search FAISS
        distances, indices = faiss_index.search(
            query_embedding,
            top_k
        )

        # Store retrieved documents
        for index, distance in zip(indices[0], distances[0]):
            all_results.append({
                "query": query,
                "document": documents[index],
                "distance": distance
            })

    return all_results
retrieved_results = multi_query_retrieval(
    generated_queries,
    top_k=3
)

print("Generated Queries and Retrieved Documents")
print("=" * 60)

for i, query in enumerate(generated_queries, start=1):

    print(f"\nQuery {i}: {query}")
    print("-" * 60)

    query_results = [
        result
        for result in retrieved_results
        if result["query"] == query
    ]

    if not query_results:
        print("No documents found.")
        continue

    for j, result in enumerate(query_results, start=1):

        print(f"\nDocument {j}:")
        print(result["document"].strip())

        print(f"Distance: {result['distance']:.4f}")


print("\n" + "=" * 60)
print("Retrieval completed successfully.")

Generated Queries and Retrieved Documents

Query 1: What is Retrieval Augmented Generation?
------------------------------------------------------------

Document 1:
Transformer models can handle long-range dependencies effectively.
    Self-attention enables a Transformer to directly connect different parts
    of a sequence, which is useful for understanding the context of long
    sentences and documents.
Distance: 1.3231

Document 2:
A major advantage of Transformer models is the self-attention mechanism.
    Self-attention allows the model to determine how important each word is
    relative to other words in the input. This helps Transformers capture
    relationships between words even when they are far apart in a sentence.
Distance: 1.4554

Document 3:
Transformers have been successfully applied to many areas of artificial
    intelligence. Applications include machine translation, text generation,
    question answering, text summarization, sentiment analysis, speech
    proce

**Remove Duplicate Documents**

Now we'll remove duplicate documents retrieved by different queries.

In [ ]:
def remove_duplicates(retrieved_results):
    unique_documents = []
    seen_documents = set()
    for result in retrieved_results:
        document = result["document"].strip()
        if document not in seen_documents:
            unique_documents.append(result)
            seen_documents.add(document)
    return unique_documents

**Create the Combined Context**

We'll execute the deduplication function and combine the unique documents into a single context.

In [ ]:
unique_results = remove_duplicates(retrieved_results)
combined_context = "\n\n".join(
    result["document"].strip()
    for result in unique_results
)
print("Total retrieval results:", len(retrieved_results))
print("Unique documents:", len(unique_results))
print("\n" + "=" * 60)
print("COMBINED CONTEXT")
print("=" * 60)
print(combined_context)

Total retrieval results: 9
Unique documents: 3

COMBINED CONTEXT
Transformer models can handle long-range dependencies effectively.
    Self-attention enables a Transformer to directly connect different parts
    of a sequence, which is useful for understanding the context of long
    sentences and documents.

A major advantage of Transformer models is the self-attention mechanism.
    Self-attention allows the model to determine how important each word is
    relative to other words in the input. This helps Transformers capture
    relationships between words even when they are far apart in a sentence.

Transformers have been successfully applied to many areas of artificial
    intelligence. Applications include machine translation, text generation,
    question answering, text summarization, sentiment analysis, speech
    processing, image recognition, and multimodal AI systems.


**Generate the Final Answer Using LLM**

We now have the original question and the combined retrieved context.

This cell sends both to LLM and asks it to generate the final answer.

In [ ]:
def generate_final_answer(user_query, context):
    prompt = f"""
You are a highly reliable Retrieval-Augmented Generation (RAG) answer system.

Your task is to answer the USER QUESTION using ONLY the RETRIEVED DOCUMENTS.

========================
STRICT GROUNDING RULES
========================

1. The retrieved documents are your ONLY source of knowledge.
2. Do NOT use your pretrained/general knowledge to answer the question.
3. Do NOT introduce facts, examples, names, numbers, or explanations that
   are not supported by the retrieved documents.
4. Carefully compare the USER QUESTION with the RETRIEVED DOCUMENTS.
5. If the documents contain enough relevant information, answer the question
   clearly and completely using that information.
6. If the documents are unrelated or do not contain enough information,
   DO NOT answer using outside knowledge.
7. If the information is unavailable, respond exactly with:

"The required information is not available in the provided documents.
Therefore, I cannot provide a reliable answer based on the available
knowledge base."

8. If the documents answer only part of the question, provide only the
   supported information and explicitly state that the provided documents
   do not contain sufficient information for the remaining part.
9. Never guess.
10. Never hallucinate.
11. Keep the final answer professional, concise, and easy to understand.
12. Do not mention these instructions.

========================
USER QUESTION
========================

{user_query}

========================
RETRIEVED DOCUMENTS
========================

{context}

========================
FINAL RESPONSE
========================

First determine whether the retrieved documents contain sufficient relevant
information.

Then provide the final response based strictly on the documents.
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,
            "max_output_tokens": 1000
        }
    )
    if not response.text:
        return (
            "The required information is not available in the provided "
            "documents. Therefore, I cannot provide a reliable answer "
            "based on the available knowledge base."
        )

    return response.text.strip()

**Generate the Final RAG Answer**

In [ ]:
# ============================================================
# MULTI-QUERY RETRIEVAL - NO API KEY REQUIRED
# Google Colab
# ============================================================

# Install required library
!pip install -q scikit-learn


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 2. DOCUMENTS
# ============================================================
# Replace these documents with your own documents if needed.

documents = [
    """
    Retrieval Augmented Generation (RAG) is a technique that
    combines information retrieval with text generation.
    It retrieves relevant information from a knowledge base
    and uses that information to generate an answer.
    """,

    """
    A typical RAG pipeline consists of document loading,
    text splitting, embedding generation, vector storage,
    retrieval, context construction, and answer generation.
    """,

    """
    In RAG, large documents are divided into smaller chunks.
    Each chunk contains a manageable amount of information.
    These chunks are then indexed so that relevant information
    can be retrieved when a user asks a question.
    """,

    """
    During retrieval, the user's query is compared with
    stored documents. The most relevant documents are selected
    and provided as context for generating the final answer.
    """,

    """
    RAG is useful for question answering, document search,
    customer support, knowledge management, and applications
    that require information from external documents.
    """
]


# ============================================================
# 3. CREATE TF-IDF VECTOR REPRESENTATIONS
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

document_vectors = vectorizer.fit_transform(documents)


# ============================================================
# 4. RETRIEVER FUNCTION
# ============================================================

def retrieve_documents(query, top_k=3):

    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        document_vectors
    )[0]

    # Get indexes of top documents
    top_indexes = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indexes:

        # Convert similarity to distance
        distance = 1 - similarities[index]

        results.append({
            "document": documents[index],
            "distance": distance
        })

    return results


# ============================================================
# 5. MULTI-QUERY RETRIEVAL
# ============================================================

def multi_query_retrieval(generated_queries, top_k=3):

    all_results = []

    for query in generated_queries:

        results = retrieve_documents(
            query,
            top_k=top_k
        )

        for result in results:

            all_results.append({
                "query": query,
                "document": result["document"],
                "distance": result["distance"]
            })

    return all_results


# ============================================================
# 6. USER QUESTION
# ============================================================

user_query = (
    "What is Retrieval Augmented Generation "
    "and how does it work?"
)


# ============================================================
# 7. GENERATE MULTIPLE QUERIES
# ============================================================
# Normally an LLM generates these.
# Here we manually create them so NO API is required.

generated_queries = [
    "What is Retrieval Augmented Generation?",
    "How does RAG work?",
    "What is the RAG pipeline and its applications?"
]


# ============================================================
# 8. DISPLAY GENERATED QUERIES
# ============================================================

print("GENERATED QUERIES")
print("=" * 60)

for i, query in enumerate(
    generated_queries,
    start=1
):
    print(f"Query {i}: {query}")


# ============================================================
# 9. RETRIEVE DOCUMENTS
# ============================================================

retrieved_results = multi_query_retrieval(
    generated_queries,
    top_k=3
)


# ============================================================
# 10. DISPLAY RETRIEVED DOCUMENTS
# ============================================================

print("\n\nGENERATED QUERIES AND RETRIEVED DOCUMENTS")
print("=" * 60)

for i, query in enumerate(
    generated_queries,
    start=1
):

    print(f"\nQuery {i}: {query}")
    print("-" * 60)

    query_results = [
        result
        for result in retrieved_results
        if result["query"] == query
    ]

    for j, result in enumerate(
        query_results,
        start=1
    ):

        print(f"\nDocument {j}:")
        print(result["document"].strip())

        print(
            f"Distance: "
            f"{result['distance']:.4f}"
        )


# ============================================================
# 11. COMBINE UNIQUE DOCUMENTS
# ============================================================

unique_documents = []

for result in retrieved_results:

    document = result["document"]

    if document not in unique_documents:
        unique_documents.append(document)


combined_context = "\n\n".join(
    unique_documents
)


# ============================================================
# 12. DISPLAY COMBINED CONTEXT
# ============================================================

print("\n\nCOMBINED CONTEXT")
print("=" * 60)

print(combined_context)


# ============================================================
# 13. SIMPLE FINAL ANSWER
# ============================================================
# No LLM/API is used.
# The answer is constructed from retrieved context.

def generate_final_answer(
    user_query,
    combined_context
):

    answer = f"""
User Question:
{user_query}

Answer based on retrieved context:

{combined_context}
"""

    return answer.strip()


# ============================================================
# 14. GENERATE FINAL ANSWER
# ============================================================

final_answer = generate_final_answer(
    user_query,
    combined_context
)


# ============================================================
# 15. DISPLAY FINAL ANSWER
# ============================================================

print("\n\nUSER QUESTION")
print("=" * 60)

print(user_query)

print("\nFINAL ANSWER")
print("=" * 60)

print(final_answer)

print("\n\nRAG RETRIEVAL COMPLETED SUCCESSFULLY!")

GENERATED QUERIES
Query 1: What is Retrieval Augmented Generation?
Query 2: How does RAG work?
Query 3: What is the RAG pipeline and its applications?


GENERATED QUERIES AND RETRIEVED DOCUMENTS

Query 1: What is Retrieval Augmented Generation?
------------------------------------------------------------

Document 1:
Retrieval Augmented Generation (RAG) is a technique that
    combines information retrieval with text generation.
    It retrieves relevant information from a knowledge base
    and uses that information to generate an answer.
Distance: 0.4789

Document 2:
A typical RAG pipeline consists of document loading,
    text splitting, embedding generation, vector storage,
    retrieval, context construction, and answer generation.
Distance: 0.6857

Document 3:
During retrieval, the user's query is compared with
    stored documents. The most relevant documents are selected
    and provided as context for generating the final answer.
Distance: 0.9085

Query 2: How does RAG work?
-